# Лекция 11: Компьютерное зрение
## Адаптеры в диффузионных моделях & SOTA модели. Часть 2

**Преподаватель:** Шевченко Елена, Т-Банк

**Аудитория:** ФКН ВШЭ, МФТИ

---

## Содержание

1. ~~Адаптеры~~
   1. ~~Зачем нужны, типы адаптеров~~
   2. ~~ControlNet~~
   3. ~~IP-Adapter~~
   4. ~~InstantStyle: инъекция по блокам~~
   5. ~~Что ещё бывает~~
   6. ~~От адаптеров к in-context conditioning~~
2. **Как устроена современная text-to-image модель** (этот ноутбук)
   0. Общая схема
   1. Flow Matching
   2. Картинка — это последовательность токенов
   3. Скалярные условия — через adaLN
   4. RoPE — позиции токенов
   5. Текст — через joint attention
   6. DoubleStream и SingleStream блоки
   7. Референсы — это тоже токены
   8. Меньше шагов
   9. Итог
3. Современные модели: FLUX и Qwen-Image
4. Дистилляция

---

Открытые модели — FLUX, Qwen-Image, SD3 — собраны из одних и тех же деталей. В этой части разбираем эти детали — восемь принципов. В 11-3 посмотрим, как из них собраны конкретные модели, в 11-4 — как их ускоряют дистилляцией.

> **Обозначения** (как в коде FLUX): $x_0$ — данные, $\epsilon \sim \mathcal{N}(0, I)$ — шум, $t = 1$ — чистый шум, $t = 0$ — чистая картинка.
> В лекции 10 наоборот: $x_0$ — шум, $x_1$ — данные.

## 0. Общая схема

![pipeline](assets/p_pipeline.png)

Любая из моделей этой лекции работает так:

1. Промпт проходит через текстовый энкодер и превращается в последовательность векторов.
2. Шум $\epsilon$ режется на патчи — получается последовательность токенов картинки.
3. Номер шага $t$ (и, если есть, guidance) превращается в один вектор условия $c$.
4. Трансформер смотрит на всё сразу и предсказывает скорость $v_\theta$ — куда сдвинуть текущий $x$.
5. Делаем шаг, повторяем N раз от $t = 1$ до $t = 0$, декодируем латент VAE-декодером.

Дальше — по одному принципу на каждый кусок схемы.

## 1. Flow Matching

![flow_diffusion](assets/flow_diffusion.png)

*На рисунке обозначения как в лекции 10: $X_0$ — шум, $X_1$ — данные. В этой части, как в коде FLUX, наоборот: $x_0$ — данные, $t = 1$ — шум.*

Почти все современные модели (SD3, FLUX, Qwen-Image) обучены как **Rectified Flow** — простейший вариант Flow Matching из лекции 10. Диффузия с предсказанием шума (SD 1.x, SDXL) — из лекции 9-2.

**Обучение.** Берём картинку $x_0$, шум $\epsilon$ и случайное $t \in [0, 1]$. Точка на прямой между ними:

$$x_t = (1 - t)\, x_0 + t\, \epsilon$$

Сеть учится предсказывать скорость движения по этой прямой — она одинакова в любой точке:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\, \big\| v_\theta(x_t, t) - (\epsilon - x_0) \big\|^2$$

**Генерация.** Стартуем из шума при $t = 1$ и идём к $t = 0$ шагами Эйлера:

$$x_{t - \Delta t} = x_t - \Delta t \cdot v_\theta(x_t, t)$$

**Почему это лучше диффузии.** На прямой с постоянной скоростью один шаг Эйлера точен. Траектории, которые выучивает сеть, всё равно немного искривлены (сеть усредняет по всем парам «картинка — шум»), но заметно прямее диффузионных. Поэтому ошибка на шаге меньше: FLUX и SD3 по умолчанию делают ~28 шагов, SDXL — 30–50. Разница не в разы; главный выигрыш — такую модель проще дистиллировать до 1–4 шагов.

Flow Matching меняет только лосс и сэмплер. Архитектура, VAE и способы подачи условия — те же, что и у диффузии.

## 2. Картинка — это последовательность токенов

Генерация идёт не в пикселях, а в латенте VAE: он в 8 раз меньше по каждой стороне. Дальше латент режется на патчи 2×2, и каждый патч линейным слоем превращается в токен — как слово в LLM.

![patchify](assets/p_patchify.png)

Сколько получится токенов, считается так ($f$ — сжатие VAE, $p$ — размер патча, $C$ — каналов латента):

$$N = \frac{H}{f\,p} \cdot \frac{W}{f\,p}, \qquad \text{размерность входа токена} = p^2 \cdot C$$

Для $1024^2$, $f = 8$, $p = 2$, $C = 16$: $64 \cdot 64 = 4096$ токенов по $4 \cdot 16 = 64$ числа.

После этого картинка для модели — просто последовательность из 4096 токенов. Отсюда три следствия:

- **Трансформер вместо U-Net.** Attention связывает любые два патча уже в первом слое; в свёрточной сети контекст растёт только с глубиной. И трансформеры предсказуемо улучшаются с ростом размера — как LLM. Эту архитектуру называют **DiT** (Diffusion Transformer).
- **Другое разрешение — другая длина.** 1024 × 512 — это 2048 токенов вместо 4096, сеть та же.
- **Текст и картинка — токены одной последовательности.** На этом построены joint attention и референсы ниже.

**Тренд в VAE.** Цена attention растёт как квадрат числа токенов. Поэтому новые VAE сжимают сильнее, но хранят больше каналов на позицию:

| | сжатие | каналов |
|---|---|---|
| SD 1.x, SDXL | ×8 | 4 |
| SD3, FLUX.1 | ×8 | 16 |
| FLUX.2, FLUX 3 | ×8 | 32 |
| Qwen-Image-2.0 | ×16 | 64 |

По формуле выше: вдвое больше $f$ — вчетверо меньше токенов на ту же картинку.

## 3. Скалярные условия — через adaLN

Есть условия, которые описываются одним числом или вектором на всю картинку: номер шага $t$, guidance scale, общий «смысл» промпта (pooled-вектор CLIP — во FLUX.1 и SD3). Их подают через **adaLN** (adaptive LayerNorm).

![dit](assets/dit.png)

На картинке — блок DiT из исходной статьи. Справа снизу **Condition**: timestep $t$ и метка класса $y$ (DiT генерировал по классу ImageNet). MLP сворачивает их в один вектор $c$. В text-to-image вместо класса — guidance (если есть) и, во FLUX.1 и SD3, pooled-вектор CLIP; у FLUX.2 и FLUX 3 pooled-вектора нет. Остальное так же.

Из $c$ блок получает шесть векторов — по три на attention и на MLP (так в DiT и в DoubleStream-блоке FLUX, на каждый поток; в SingleStream attention и MLP идут параллельно, и векторов три):
- $\gamma$ (Scale) и $\beta$ (Shift) — применяются сразу после Layer Norm;
- $\alpha$ (Scale над подблоком) — множитель выхода перед сложением с residual ($\oplus$).

Для attention (индекс 1; для MLP то же с индексом 2):

$$h = \gamma_1(c) \odot \text{LN}(x) + \beta_1(c)$$

$$x \leftarrow x + \alpha_1(c) \odot \text{Attention}(h)$$

LN здесь без своих обучаемых параметров — их роль играют $\gamma$ и $\beta$.

Как читать:
- **$\gamma$ и $\beta$** — условие растягивает и сдвигает признаки. Например, на шумном шаге и на почти чистом сеть должна вести себя по-разному.
- **$\alpha$** — сколько этот подблок добавляет к residual.

В коде DiT и FLUX те же векторы называются scale, shift, gate, а вместо $\gamma$ пишут $(1 + \text{scale})$: при нулевом scale нормализация остаётся обычной.

**adaLN-Zero** (так в DiT): последний слой MLP, который выдаёт эти векторы, инициализируют нулями. Тогда shift = scale = gate = 0, то есть $\gamma = 1$, $\beta = 0$, $\alpha = 0$ и каждый блок на старте — тождественное отображение. Сеть начинает как «пустая» и постепенно включает блоки. Та же идея, что zero conv в ControlNet.

**Важно:** $c$ одинаковый для всех токенов. Он говорит сети «какой сейчас шаг и какой стиль в целом», но не «что нарисовать в левом углу». Что и где рисовать, модель узнаёт из текстовых токенов через attention.

## 4. RoPE — позиции токенов

Трансформер обрабатывает токены без понятия о порядке — self-attention симметричен. Нужен способ сообщить модели, где находится каждый токен.

Классическое решение — добавить позиционный вектор к токену:

$$\text{input}_i = \text{token}_i + \text{pos\_emb}_i$$

Проблема: позиции фиксированы при обучении. При другом разрешении модель встречает незнакомые позиции — плохая экстраполяция.

RoPE решает это иначе. Вместо того чтобы добавлять позицию к токену, RoPE вращает векторы запросов $Q$ и ключей $K$ перед вычислением attention. Угол вращения зависит от позиции токена.

Ключевое свойство: скалярное произведение после вращения зависит от самих векторов $q$, $k$, а от позиций — только через относительное расстояние между токенами:

$$q_m^T k_n = g(q, k, m - n)$$

```
позиции m=5,   n=3   → зависит от 5-3=2
позиции m=105, n=103 → зависит от 105-103=2

результат одинаковый
```

Для изображений RoPE расширяется на 2D — вектор делится на две половины, каждая вращается по своей оси:

$$\text{вектор размерности } d = \underbrace{d/2}_{\text{ось X (столбец)}} + \underbrace{d/2}_{\text{ось Y (строка)}}$$

Это и даёт гибкость к разрешению:

![rope](assets/p_rope.png)

```
обучение:  1024×1024 → токены в сетке [0..63]²
генерация: 2048×1024 → токены в сетке [0..127]×[0..63]

столбцов 64..127 модель не видела,
но RoPE кодирует относительные расстояния
→ "этот токен на 3 столбца правее соседнего"
→ работает и на новом разрешении
```

Ось можно добавить и третью — например, номер изображения. Это понадобится для референсов.

## 5. Текст — через joint attention

![attention](assets/p_attention.png)

**Как было: cross-attention** (SD 1.x, SDXL). $Q$ — из токенов картинки, $K$ и $V$ — из текста. Текстовые векторы одинаковы во всех слоях и картинку не видят: информация идёт только от текста к картинке.

**Как стало: joint attention** (MM-DiT, впервые в SD3). Пусть $e$ — $n$ текстовых токенов, $x$ — $m$ токенов картинки.

1. У каждой модальности свои проекции: $Q_e = e\, W_Q^{\text{txt}}$, $Q_x = x\, W_Q^{\text{img}}$, так же для $K$ и $V$.
2. Проекции склеиваются по длине: $Q = [Q_e; Q_x]$, $K = [K_e; K_x]$, $V = [V_e; V_x]$.
3. Один attention над $n + m$ токенами: $\text{softmax}(QK^\top / \sqrt{d})\, V$.
4. Выход режется обратно на текст и картинку, у каждой — свой MLP.

Что это даёт:
- **обмен в обе стороны**: текстовые токены от слоя к слою уточняются с учётом того, что уже нарисовано;
- **свои веса** для текста и картинки — у модальностей разная статистика;
- **цена**: attention квадратичен по общей длине — 4096 токенов картинки + 512 текста = 4608.

![mm-dit](assets/mm-dit.png)

**Откуда берутся текстовые токены.** Раньше — T5 (последовательность токенов) плюс CLIP (один pooled-вектор для adaLN). Сейчас — большая мультимодальная LLM (VLM): Mistral во FLUX.2, Qwen-VL в Qwen-Image и FLUX 3. Во FLUX.2 и FLUX 3 берут не только последний слой, а скрытые состояния нескольких слоёв сразу:

![text_encoder](assets/p_text_encoder.png)

VLM понимает и текст, и картинки, поэтому через неё же можно подать референсное изображение «на словах» — так делает Qwen-Image-Edit.

## 6. DoubleStream и SingleStream блоки

Joint attention можно устроить двумя способами:

![blocks](assets/p_blocks.png)

- **DoubleStream** — у текста и картинки свои проекции и MLP, общий только attention. Модальности остаются разными.
- **SingleStream** — текст и картинка уже одна последовательность с общими весами. Блок дешевле и проще. Attention и MLP считаются параллельно от одного входа:

$$h = \gamma \odot \text{LN}(x) + \beta, \qquad x \leftarrow x + \alpha \odot \big(W_1\,\text{Attn}(h) + W_2\,\text{MLP}(h)\big)$$

  В обычном блоке MLP идёт после attention — это два последовательных residual.

Обычно сначала идут DoubleStream-блоки (каждая модальность строит своё представление), потом SingleStream (общая обработка). Баланс у моделей разный: в семействе FLUX он со временем смещается к общим блокам, а у Qwen-Image все 60 блоков — DoubleStream:

![block_counts](assets/p_block_counts.png)

У FLUX 3 Klein первые блоки ещё более раздельные: внутри каждого потока (текст, картинка, референсы) — свой attention, без обмена. Обмен начинается только в 28 общих блоках.

**QK-norm.** Во всех этих моделях внутри каждой головы $q$ и $k$ нормализуют RMSNorm перед скалярным произведением:

$$\hat q = \frac{q}{\sqrt{\frac{1}{d}\sum_i q_i^2 + \varepsilon}} \odot g_q, \qquad \hat k = \frac{k}{\sqrt{\frac{1}{d}\sum_i k_i^2 + \varepsilon}} \odot g_k, \qquad \text{score} = \frac{\hat q^\top \hat k}{\sqrt d}$$

где $g_q, g_k \in \mathbb{R}^d$ — обучаемые векторы масштаба (инициализируются единицами), $\odot$ — поэлементное умножение.

Зачем:
- без нормализации нормы $q$ и $k$ в больших моделях растут, вместе с ними растёт score;
- softmax от больших чисел — почти one-hot: градиенты через него пропадают, в fp16 — переполнения, обучение срывается;
- после RMSNorm $\|\hat q\|, \|\hat k\| \approx \sqrt d$, и score ограничен. Насколько «острым» будет внимание, решают обучаемые $g_q, g_k$, а не случайный рост весов.

## 7. Референсы — это тоже токены

Редактирование («сделай этот кадр зимним») и генерация по референсу раньше требовали адаптеров — ControlNet, IP-Adapter (лекция 11-1). В трансформере проще: референсную картинку прогоняем через тот же VAE, режем на патчи и дописываем её токены в ту же последовательность. Это называют **in-context conditioning**.

![references](assets/p_references.png)

Чтобы модель отличала референс от генерируемой картинки, достаточно трёх вещей:

1. **Номер изображения в RoPE.** К двум осям $(h, w)$ добавляется третья: генерируемая картинка — 0, референсы — 1, 2, …

   $$\text{pos}_{\text{target}} = (0, h, w), \qquad \text{pos}_{\text{ref}_k} = (k, h, w)$$

   Это схема; ровно так — во FLUX.1 Kontext (один референс, $k = 1$). Во FLUX.2 номер — сдвиг по той же оси на $10(k+1)$, во FLUX 3 — отдельная четвёртая ось со значением $-(k+1)$ (подробно — в 11-3).

   Пиксель $(h, w)$ референса и тот же $(h, w)$ результата совпадают по месту, но различаются номером. Поэтому модель легко копирует то, что не надо менять.
2. **Референс не зашумляется.** Он всегда чистый, зашумлена только генерируемая картинка.
3. **Лосс только по генерируемой картинке.** Референсные токены лишь подсказывают:

   $$\mathcal{L} = \big\| v_\theta([\,e;\ x_t;\ y_{\text{ref}}\,], t)_{\text{target}} - (\epsilon - x_0) \big\|^2$$

   где $e$ — текстовые токены, $x_t$ — зашумлённая картинка, $y_{\text{ref}}$ — чистые референсы, индекс target — берём только выходы на токенах картинки.

Новых модулей нет — работает обычный joint attention. По этой схеме сделаны FLUX.1 Kontext, Qwen-Image-Edit, FLUX.2 и FLUX 3; детали (как именно кодируется номер, какой timestep у референсов) различаются — в 11-3.

## 8. Меньше шагов

Обычная модель делает 20–50 шагов, и с CFG каждый шаг — два прохода сети (с условием и без). Это дорого, поэтому почти у каждой модели есть быстрая версия:

| модель | шагов | как |
|---|---|---|
| FLUX.1 [dev] | 20–50 | guidance distillation: CFG «вшит» в сеть, один проход на шаг |
| FLUX.1 [schnell] | 1–4 | adversarial distillation (LADD) |
| FLUX 3 Klein distilled | 16 | flow map: сеть учится прыгать сразу на много шагов, без CFG |
| Qwen-Image-2.0 | 4 | DMD — distribution matching distillation |

Как это работает — в 11-4.

## 9. Итог

| принцип | что решает | где видно в схеме |
|---|---|---|
| Flow Matching | прямые траектории: меньше ошибка на шаге, проще дистиллировать | шаг Эйлера |
| картинка = токены | трансформер вместо U-Net, любое разрешение | патчи → токены |
| adaLN | шаг, guidance и общий стиль влияют на каждый блок | вектор $c$ |
| RoPE | позиции через относительные смещения | внутри attention |
| joint attention | текст и картинка обмениваются в обе стороны | токены текста |
| Double / SingleStream | сначала своё представление, потом общее | внутри трансформера |
| референсы = токены | редактирование без адаптеров | дописанные токены |
| меньше шагов | быстрая генерация | число итераций |

Дальше, в 11-3: FLUX.1 как эталон и как из этих принципов собраны Kontext, FLUX.2, FLUX 3 и Qwen-Image.